# PPO Training Smoke Test
> This notebook currently uses `MockDetector` only for integration testing. Final experimental results must use a real deepfake detector provided through the detector adapter interface.

## 1. Imports

In [1]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from PIL import Image

from src.ppo.environment import DeepfakeAttackEnv
from src.ppo.mock_detector import MockDetector
from src.ppo.train import create_ppo_model, load_ppo, run_episode, save_ppo, train_ppo

## 2. Create the mock-backed environment

In [2]:
height, width = 96, 128
x = np.linspace(48, 240, width, dtype=np.uint8)
gradient = np.tile(x, (height, 1))
image = Image.fromarray(np.stack([gradient] * 3, axis=2)).convert('RGB')
env = DeepfakeAttackEnv(image, MockDetector(), max_steps=5, seed=42)

## 3. Create PPO

In [3]:
model = create_ppo_model(env, seed=42)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


## 4. Train for 1,000 requested timesteps

In [4]:
model = train_ppo(model, total_timesteps=1000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 4.99     |
|    ep_rew_mean     | -0.0044  |
| time/              |          |
|    fps             | 2737     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------


## 5. Save the checkpoint

In [5]:
checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'ppo_mock_v1'
save_ppo(model, checkpoint_path)
print(checkpoint_path.with_suffix('.zip'))

/Users/hgtan/Desktop/DimSum/checkpoints/ppo_mock_v1.zip


## 6. Reload the checkpoint

In [6]:
loaded_model = load_ppo(checkpoint_path, env)
print('Checkpoint reloaded.')

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Checkpoint reloaded.


## 7. Run and save one deterministic episode

In [7]:
history = run_episode(loaded_model, env, deterministic=True)
columns = ['step', 'action_name', 'before_confidence', 'after_confidence', 'reward', 'success']
for row in history:
    print({column: row[column] for column in columns})

results_path = PROJECT_ROOT / 'results' / 'ppo_mock_episode.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
with results_path.open('w', newline='') as output_file:
    writer = csv.DictWriter(output_file, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)
print(results_path)

{'step': 1, 'action_name': 'none', 'before_confidence': 0.547500214073807, 'after_confidence': 0.547500214073807, 'reward': 0.0, 'success': False}
{'step': 2, 'action_name': 'none', 'before_confidence': 0.547500214073807, 'after_confidence': 0.547500214073807, 'reward': 0.0, 'success': False}
{'step': 3, 'action_name': 'none', 'before_confidence': 0.547500214073807, 'after_confidence': 0.547500214073807, 'reward': 0.0, 'success': False}
{'step': 4, 'action_name': 'none', 'before_confidence': 0.547500214073807, 'after_confidence': 0.547500214073807, 'reward': 0.0, 'success': False}
{'step': 5, 'action_name': 'resize', 'before_confidence': 0.547500214073807, 'after_confidence': 0.5475002557970583, 'reward': -4.1723251298364517e-08, 'success': False}
/Users/hgtan/Desktop/DimSum/results/ppo_mock_episode.csv
